# DATA 612 — Benchmark data prep: Taobao UserBehavior (3h windows)

Converts Taobao UserBehavior (Alibaba, ~100M events, 9 days) into **exactly the same tables, labels and splits
as the Criteo pipeline**, using the repo's own code (`src/data/preprocess.py`, `src/data/splits.py`,
`scripts/prepare_criteo.py`).

**Mapping (Criteo → Taobao)**

| Criteo | Taobao UserBehavior |
|---|---|
| user (`uid`) | `user_id` |
| impression row | any behaviour (pv, cart, fav, buy) |
| campaign (token) | item **category**, top `MAX_CATEGORIES` + `other` |
| click | cart, fav or buy |
| conversion (label) | **buy** |
| cost | not available → 0 |

**Window length: 3h** (not 12h). The log covers only 9 days (25 Nov – 3 Dec 2017, China time); 12h windows
would give just 18 windows and a 2–3 window test set. 3h gives 72 windows. Windows start at midnight China time.
Rows outside the official 9-day range (the raw file has a few) are dropped and counted.

Everything else is identical to Criteo: predict window t+1 from windows ≤ t, 7-day activity rule,
chronological 70/15/15 by label window, hashed 70/15/15 user groups.

**Runtime:** CPU **with High-RAM** (or an A100 runtime, which has more RAM). This is the largest benchmark.
If memory runs out, set `USER_FRACTION = 0.5` (or lower): a fixed, hashed subset of users is kept.

## 1. Settings

In [ ]:
import os, sys, json, time, shutil, subprocess, importlib.util, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/DATA612_data")
    REPO_DIR  = Path("/content/repo")
    WORK_DIR  = Path("/content/work")
else:
    DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", "drive_sim/DATA612_data")).resolve()
    REPO_DIR  = Path(os.environ.get("REPO_DIR", "repo")).resolve()
    WORK_DIR  = Path(os.environ.get("WORK_DIR", "work")).resolve()

WINDOW_HOURS   = 3
MAX_CATEGORIES = 675        # same vocabulary size as Criteo's 675 campaigns; rarer categories -> "other"
USER_FRACTION  = 1.0        # 1.0 = all users; lower (e.g. 0.5) only if RAM runs out
N_SHARDS       = 16
# Official range: 2017-11-25 00:00 to 2017-12-04 00:00, China time (UTC+8)
START_TS, END_TS = 1511539200, 1512316800
RAW_DIR = DRIVE_DIR / "raw" / "taobao"
NAME    = f"taobao_w{WINDOW_HOURS:g}h" + ("" if USER_FRACTION >= 1 else f"_u{int(USER_FRACTION * 100)}")
RAW_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Raw files:", RAW_DIR, "| output name:", NAME)

## 2. Repo (the shared pipeline) and packages

In [ ]:
REPO_URL = "https://github.com/Abhyudaya01/criteo-customer-journey-transformers.git"
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
elif IN_COLAB:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "scripts"))

need = [p for p, m in {"pyarrow": "pyarrow"}.items() if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import numpy as np, pandas as pd, yaml
from src.data.preprocess import DataColumns, build_campaign_vocab, build_windowed_dataset, finalize_meta, \
    n_windows_for, write_table
from src.data.splits import window_boundaries
import prepare_criteo as pc          # reuse: add_split_columns, split_counts, merge_counts, shard_ids, TABLES
cfg = yaml.safe_load((REPO_DIR / "configs" / "base.yaml").read_text())
SEED = cfg["project"].get("seed", 42)
print("pandas", pd.__version__, "| seed", SEED)

## 3. Raw file (manual download, once)
Tianchi needs a (free) Alibaba Cloud login, so this step is manual:
1. Open https://tianchi.aliyun.com/dataset/649 , log in, and download **UserBehavior.csv.zip** (~0.9 GB).
2. Upload it to Google Drive at `MyDrive/DATA612_data/raw/taobao/` (the unzipped `UserBehavior.csv` also works).
3. Run this cell. It copies the file to local disk (much faster than reading from Drive).

In [ ]:
candidates = [RAW_DIR / "UserBehavior.csv", RAW_DIR / "UserBehavior.csv.zip"]
src = next((p for p in candidates if p.exists()), None)
assert src is not None, f"Put UserBehavior.csv.zip (or UserBehavior.csv) in {RAW_DIR} first (see the steps above)."
local_csv = WORK_DIR / "UserBehavior.csv"
if not local_csv.exists():
    t0 = time.time()
    if src.suffix == ".zip":
        with zipfile.ZipFile(src) as z:
            member = next(n for n in z.namelist() if n.endswith(".csv"))
            with z.open(member) as fin, open(local_csv, "wb") as fout:
                shutil.copyfileobj(fin, fout, length=16 << 20)
    else:
        shutil.copy2(src, local_csv)
    print(f"Copied to local disk in {time.time() - t0:.0f}s")
print(f"{local_csv} ({local_csv.stat().st_size / 1e9:.2f} GB)")

## 4. Map Taobao behaviours to the Criteo input schema

In [ ]:
import pyarrow as pa, pyarrow.csv as pacsv
t0 = time.time()
tbl = pacsv.read_csv(local_csv,
    read_options=pacsv.ReadOptions(column_names=["user_id", "item_id", "category_id", "behavior", "timestamp"]),
    convert_options=pacsv.ConvertOptions(column_types={"user_id": pa.int64(), "item_id": pa.int64(),
        "category_id": pa.int64(), "behavior": pa.dictionary(pa.int32(), pa.string()), "timestamp": pa.int64()}))
ev = tbl.to_pandas()
del tbl
N_RAW = len(ev)
print(f"read {N_RAW:,} rows in {time.time() - t0:.0f}s (official file: 100,150,807 rows)")

in_range = (ev["timestamp"] >= START_TS) & (ev["timestamp"] < END_TS)
N_OUT_OF_RANGE = int((~in_range).sum())
ev = ev[in_range]
if USER_FRACTION < 1:
    h = pd.util.hash_array(ev["user_id"].to_numpy(), hash_key="taobao-user-subs", categorize=False)
    ev = ev[(h % np.uint64(10_000)) < int(USER_FRACTION * 10_000)]
ev = ev.sort_values(["timestamp", "user_id"], kind="mergesort").reset_index(drop=True)
print(f"dropped {N_OUT_OF_RANGE:,} rows outside 25 Nov - 3 Dec 2017 | kept {len(ev):,} rows")
print(f"users: {ev['user_id'].nunique():,} | items: {ev['item_id'].nunique():,} | categories: {ev['category_id'].nunique():,}")
print("behaviours:", ev["behavior"].value_counts().to_dict())

counts = ev["category_id"].value_counts()
keep = counts.index[:MAX_CATEGORIES - 1].to_numpy()
token = np.where(np.isin(ev["category_id"].to_numpy(), keep), ev["category_id"].to_numpy(), -2)   # -2 = other
print(f"categories: {len(counts):,} -> tokens: {len(np.unique(token))} (share of events in 'other': {(token == -2).mean():.1%})")

bcat = ev["behavior"].astype("category")                 # compact codes (no 100M Python strings)
bcode, bnames = bcat.cat.codes.to_numpy(), [str(c) for c in bcat.cat.categories]
code_of = lambda names: [bnames.index(n) for n in names if n in bnames]
is_buy = np.isin(bcode, code_of(["buy"]))
ts = (ev["timestamp"].to_numpy(np.int64) - START_TS)                           # seconds from midnight 25 Nov (UTC+8)
raw = pd.DataFrame({
    "timestamp": ts,
    "uid": ev["user_id"].to_numpy(np.int64),
    "campaign": token.astype(np.int64),
    "conversion": is_buy.astype(np.int8),
    "conversion_timestamp": np.where(is_buy, ts, -1).astype(np.int64),           # a purchase converts at its own time
    "conversion_id": np.where(is_buy, np.arange(len(ev)), -1).astype(np.int64),  # each buy row is one purchase
    "click": np.isin(bcode, code_of(["cart", "fav", "buy"])).astype(np.int8),
    "cost": np.zeros(len(ev), dtype=np.float32),
})
del ev, bcat, bcode, token, is_buy, ts
print(f"mapped to Criteo schema in {time.time() - t0:.0f}s:", raw.shape)
raw.head()

## 5. Build windows, labels and splits (same code path as `scripts/prepare_criteo.py`)

In [ ]:
cols = DataColumns.from_config(cfg["data"])
win = cfg["windowing"]
horizon = int(win.get("horizon_windows", 1))
max_inactive = win.get("max_inactive_hours", 168)
s_cfg = cfg["splits"]

vocab = build_campaign_vocab(raw, cols)
shards = pc.shard_ids(raw[cols.user], N_SHARDS, SEED)
n_windows = n_windows_for(raw, WINDOW_HOURS, cols)
bounds = window_boundaries(n_windows, s_cfg.get("train_frac", 0.70), s_cfg.get("val_frac", 0.15), horizon)
out_dir = WORK_DIR / "processed" / NAME
if out_dir.exists():
    shutil.rmtree(out_dir)
write_table(vocab, out_dir / "campaign_vocab.parquet")

t0 = time.time()
totals, counts, meta = {k: 0 for k in pc.SUMMED_META}, {}, {}
for i in range(N_SHARDS):
    part = raw[shards == i]
    if part.empty:
        continue
    res = build_windowed_dataset(part, WINDOW_HOURS, max_inactive, horizon, cols,
                                 vocab=vocab, n_windows=n_windows, timestamp_unit="seconds")
    res["positions"] = pc.add_split_columns(res["positions"], bounds, cfg, SEED)
    counts = pc.merge_counts(counts, pc.split_counts(res["positions"]))
    for t in pc.TABLES:
        write_table(res[t], out_dir / t / f"part-{i:03d}.parquet")
    meta = res["meta"]
    for k in pc.SUMMED_META:
        totals[k] += meta[k]
    print(f"  shard {i + 1}/{N_SHARDS}: {meta['n_positions']:,} positions")

meta.update(totals)
meta["n_campaigns"] = int(len(vocab))
meta = finalize_meta(meta)
meta["splits"] = {"boundaries": bounds.as_dict(), **counts}
meta.update({"seed": SEED, "n_shards": N_SHARDS, "source_dataset": "taobao_userbehavior",
             "campaign_token": f"item category (top {MAX_CATEGORIES - 2} + other + unknown)",
             "conversion": "buy", "click": "cart, fav or buy", "windows_per_day": int(24 // WINDOW_HOURS), "user_fraction": USER_FRACTION, "rows_dropped_out_of_range": int(N_OUT_OF_RANGE), "cost": "not available (0)"})
(out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
print(f"built in {time.time() - t0:.0f}s")

## 6. Sanity checks

In [ ]:
pos = pd.read_parquet(out_dir / "positions")
summary = {
    "window hours": WINDOW_HOURS, "windows": meta["n_windows"], "events": meta["n_impressions"],
    "visitors": meta["n_users"], "tokens (categories)": meta["n_campaigns"],
    "distinct transactions": meta["n_conversions_total"], "positions": meta["n_positions"],
    "positive positions": meta["n_positive_positions"],
    "positive rate": f"{meta['positive_rate_conversion']:.4%}",
    "positions with a next-window category": f"{meta['frac_positions_with_next_campaign']:.1%}",
    "conversion windows covered": f"{meta['frac_conversion_windows_covered']:.1%}",
}
display(pd.Series(summary, name=NAME).to_frame())
print("Split boundaries (label windows):", meta["splits"]["boundaries"])
display(pd.DataFrame(meta["splits"]["time_split"]).T.rename_axis("time_split"))
display(pd.DataFrame(meta["splits"]["user_group"]).T.rename_axis("user_group"))

checks = {
    "labels come from the next window": bool((pos["label_window"] == pos["window"] + horizon).all()),
    "no position beyond the last window": bool(pos["label_window"].max() <= meta["n_windows"] - 1),
    "positives exist in every split": all(meta["splits"]["time_split"][s]["n_positive"] > 0 for s in ("train", "val", "test")),
    "same columns as Criteo positions": set(pos.columns) >= {"uid", "window", "label_window", "is_active",
        "windows_since_active", "y_conversion", "n_next_campaigns", "time_split", "user_group"},
}
for k, v in checks.items():
    print(("OK      " if v else "PROBLEM ") + k)

## 7. Copy to Drive

In [ ]:
dst = DRIVE_DIR / "processed" / NAME
t0 = time.time()
shutil.copytree(out_dir, dst, dirs_exist_ok=True)
size = sum(f.stat().st_size for f in dst.rglob("*") if f.is_file()) / 1e9
print(f"{NAME} -> {dst} ({size:.2f} GB, {time.time() - t0:.0f}s)")
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive synced and unmounted.")
print('Train on it later with DATASET = "' + NAME + '" in any training notebook.')

## Notes for the report
- Taobao has **no ad spend or campaigns**; item categories play the role of campaigns, and purchases play the role
  of conversions. It tests whether the models generalise beyond advertising data.
- **3h windows** (not 12h) because the log covers only 9 days. The training notebooks' calendar features assume
  12h windows; they will be adapted (time of day from `windows_per_day` in `meta.json`) before benchmark training.
- Rows outside the official 9-day range are dropped (count in `meta.json`).
- If `USER_FRACTION < 1` was used, say so: the output folder name ends in `_u<percent>`.